In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

# Progress bar
from tqdm.auto import tqdm
tqdm.pandas()

pd.set_option('display.max_columns', None)

In [3]:
from news_featuring import extract_news_features_pipeline
from news_featuring import floor_or_ceil
from news_featuring import aggregate_events

# Load and preprocess events

> Upload

In [4]:
PATH_TO_NEWS = 'economic_events/economic_events.csv'
news = pd.read_csv(PATH_TO_NEWS)
news = news[['date', 'currency', 'importance', 'title', 'indicator', 'country', 'category', 'actual', 'forecast', 'previous', 
             'comment', 'source', 'source_url', 'referenceDate', 'period', 'unit', 'scale']]
news['date'] = pd.to_datetime(news['date'], utc=True)
news.sort_values(by='date', ascending=True, inplace=True)

In [5]:
print(f'Start date: {news['date'].dt.date.min()}')
print(f'End date: {news['date'].dt.date.max()}')
print(f'Count of rows: {news.shape[0]}')
print(f'Shape: {news.shape}')

Start date: 2013-01-04
End date: 2026-03-30
Count of rows: 99029
Shape: (99029, 17)


In [6]:
def preprocess_events(news: pd.DataFrame, datetime_crop_method: str = '1st'):
    news = extract_news_features_pipeline(news)

    print('[INFO] Cropping datetime hours...')
    if datetime_crop_method == '1st':
        news['date'] = pd.to_datetime(news['date'], utc=True)
        news['rounded_time'] = news['date'].apply(lambda x: x.floor('1h'))

    elif datetime_crop_method == '2nd':
        news['date'] = pd.to_datetime(news['date'], utc=True)
        news['rounded_time'] = news['date'].apply(lambda x: floor_or_ceil(x, freq='h'))
    
    print('[INFO] Aggregating events...')
    agg_events = aggregate_events(news, dt_col='rounded_time')
    agg_events['time_to_check'] = agg_events['rounded_time'] - pd.Timedelta(hours=1)
    print('[INFO] Done!')

    return agg_events

In [7]:
agg_events = preprocess_events(news, '1st')

[INFO] Category dummies added.
[INFO] Currency dummies added.
[INFO] Country dummies added.
[INFO] Source dummies added.
[INFO] Event category dummies added.
[INFO] Stage release dummies added.
[INFO] Event calculation period dummies added.
[INFO] Scale dummies added.
[INFO] Most important event dummies added.
[INFO] Flag "is_calendar" added.
[INFO] Flag "is_president" added.
[INFO] Flag "is_election" added.
[INFO] Removed "OTHER" and redundant columns.
[INFO] Cropping datetime hours...
[INFO] Aggregating events...
[INFO] Done!


In [8]:
agg_events


,rounded_time,news_count,high_impact_count,key_event_count,main_event,prev_hour_news_count,next_hour_news_count,prev_hour_high_impact_count,next_hour_high_impact_count,prev_hour_main_event,next_hour_main_event,category_bnd,category_bsnss,category_cnsm,category_enrg,category_gdp,category_gov,category_hse,category_lbr,category_mny,category_mrkt,category_prce,category_trd,currency_AUD,currency_CAD,currency_EUR,currency_GBP,currency_JPY,currency_SEK,currency_SGD,currency_USD,country_AU,country_CA,country_DE,country_EU,country_FR,country_GB,country_JP,country_SE,country_SG,country_US,source_CENTRAL_BANK,source_ENERGY,source_GOV_MINISTRY,source_INDUSTRY,source_OFFICES,source_OFFICIAL_STATS,source_PRIVATE_SURVEY,source_RATING_AGENCY,source_RESEARCH_INSTITUTES,source_US_HOUSING,event_MONETARY_POLICY,event_CB_SPEECH,event_INFLATION,event_LABOR_MARKET,event_ECONOMIC_ACTIVITY,event_SENTIMENT,event_CONSUMER_HOUSING,event_TRADE_FINANCE,event_COMMODITIES,stage_release_Final,stage_release_Flash,stage_release_Preliminary,calc_period_MoM,calc_period_QoQ,calc_period_YoY,is_calendar,is_president,mie_Balance_of_Trade,mie_Core_Inflation_rate,mie_FOMC,mie_GDP,mie_Inflation_rate,mie_Interest_Rate_Decision,mie_NFP,mie_PMI,mie_PMI_Manufacturing,mie_PMI_Services,mie_Retail_Sales,mie_Unemployment_rate,time_passed_from_last_events,time_left_to_next_events,last_important_event_in_hours,time_to_check
0,2013-01-04 13:00:00+00:00,2,1,2,NFP,0.0,1.0,0.0,0.0,None,Inflation_rate,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,2,0,0,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0.0,96.0,NaN,2013-01-04 12:00:00+00:00
1,2013-01-06 13:00:00+00:00,1,0,1,Inflation_rate,2.0,5.0,1.0,0.0,NFP,Retail_Sales,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,96.0,520.0,48.0,2013-01-06 12:00:00+00:00
2,2013-01-17 09:00:00+00:00,5,0,5,Retail_Sales,1.0,27.0,0.0,0.0,Inflation_rate,Unemployment_rate,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,10,0,0,0,0,0,2,0,3,0,0,0,0,0,0,0,0,0,0,0,0,5,0,520.0,288.0,260.0,2013-01-17 08:00:00+00:00
3,2013-01-23 09:00:00+00:00,27,0,1,Unemployment_rate,5.0,1.0,0.0,0.0,Retail_Sales,No main events,0,2,0,0,0,11,0,1,13,0,0,0,0,0,10,3,0,4,0,10,0,0,0,10,0,3,0,4,0,10,13,0,0,0,0,1,0,0,0,0,15,16,0,2,0,0,2,2,0,0,0,0,0,0,0,7,1,0,0,0,0,0,0,0,0,0,0,0,1,288.0,104.0,144.0,2013-01-23 08:00:00+00:00
4,2013-01-25 13:00:00+00:00,1,0,0,No main events,27.0,1.0,0.0,0.0,Unemployment_rate,No main events,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,104.0,170.0,52.0,2013-01-25 12:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33559,2026-03-30 13:00:00+00:00,4,0,0,No main events,4.0,2.0,1.0,0.0,Inflation_rate,No main events,3,0,0,0,0,1,0,0,0,0,0,0,0,0,3,0,0,0,0,1,0,0,0,0,3,0,0,0,0,1,0,0,3,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,1.0,2026-03-30 12:00:00+00:00
33560,2026-03-30 14:00:00+00:00,2,0,0,No main events,4.0,2.0,0.0,0.0,No main events,No main events,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,2,3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,2.0,1.0,2026-03-30 13:00:00+00:00
33561,2026-03-30 15:00:00+00:00,2,0,0,No main events,2.0,1.0,0.0,0.0,No main events,No main events,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.0,10.0,1.0,2026-03-30 14:00:00+00:00
33562,2026-03-30 20:00:00+00:00,1,0,0,No main events,2.0,10.0,0.0,0.0,No main e

## Load prices

In [9]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

path = os.path.join('..', 'dataset', 'mt5_tw_h1')
files = os.listdir(path)
pairs = [p.split('_')[0] for p in files]

In [10]:
from price_featuring import add_features, get_base_and_quote_currency
from targets import set_targets

In [11]:
def add_features_and_targets_to_prices(prices: pd.DataFrame, period: int, N: int = 8):
    prices = add_features(prices, period=period)
    prices = set_targets(prices, look_forward_bars=N)
    return prices

In [12]:
PERIOD = 21
LOOK_FORWARD = 4

> Load and preprocessing prices

In [13]:
print(pairs)

['AUDCAD', 'AUDUSD', 'EURAUD', 'EURCAD', 'EURGBP', 'EURUSD', 'GBPCHF', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']


In [14]:
prices = []
for pair in (pbar := tqdm(pairs, desc="Reading files", total=len(pairs))):
    pbar.set_description(f"Reading file: {pair}")
    pair_prices = pd.read_csv(os.path.join(path, f'{pair}_H1.csv'))
    pair_prices.rename(columns={'datetime': 'time'}, inplace=True)
    pair_prices['instrument'] = pair
    pair_prices[['base_currency', 'quote_currency']] = get_base_and_quote_currency(pair)
    pair_prices = add_features_and_targets_to_prices(pair_prices, period=PERIOD, N=8)
    pair_prices.drop(['open', 'high', 'low', 'close', 'realized_vol_long', 'realized_vol_short'], axis=1, inplace=True)
    prices.append(pair_prices)

prices = pd.concat(prices, ignore_index=True)

Reading file: USDJPY: 100%|██████████| 12/12 [00:04<00:00,  2.70it/s]


In [15]:
prices.shape

(1188197, 49)

In [16]:
prices.head()

,time,instrument,base_currency,quote_currency,daily_atr,trange,atr,kaufman_efficiency_ratio,custom_efficiency_ratio,wick_ratio,relative_range,relative_atr,normalized_bb_width,distance_from_sma,ADX_21,ADXR_21_2,DMP_21,DMN_21,di_spread,tr_over_atr,realized_vol_ratio,parkinson_vol,parkinson_vol_over_atr,atr_short_over_long,abs_log_return_sum,vvov,week,month,quarter,dayofweek,day,hour,log_return,trg_future_range_1h,trg_future_range_3h,trg_future_range_6h,trg_future_range_24h,trg_overall_future_range_1h,trg_overall_future_range_3h,trg_overall_future_range_6h,trg_overall_future_range_24h,trg_big_doji,trg_dir_changes,trg_is_flat_flg,trg_is_trend_flg,trg_is_chaos_1h,trg_is_chaos_3h,trg_is_chaos_6h,trg_is_chaos_24h
0,2010-05-07 22:00:00+00:00,AUDCAD,AUD,CAD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18,5,2,4,7,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0,0,False,False,False,False
1,2010-05-10 00:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00319,NaN,NaN,NaN,0.087774,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,0,0.001251,NaN,NaN,NaN,NaN,1.543815,NaN,NaN,NaN,0,2.0,0,0,False,False,False,False
2,2010-05-10 01:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00183,NaN,NaN,NaN,0.311475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,1,0.000636,NaN,NaN,NaN,NaN,1.741669,NaN,NaN,NaN,0,2.0,0,0,True,False,False,False
3,2010-05-10 02:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00305,NaN,NaN,NaN,0.475410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,2,-0.001596,NaN,NaN,NaN,NaN,1.646529,NaN,NaN,NaN,0,1.0,0,0,True,False,False,False
4,2010-05-10 03:00:00+00:00,AUDCAD,AUD,CAD,NaN,0.00411,NaN,NaN,NaN,0.615572,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,5,2,0,10,3,0.002737,NaN,NaN,NaN,NaN,1.541711,NaN,NaN,NaN,0,0.0,0,0,False,False,False,False


## Add log features

In [17]:
for trg in tqdm(['trg_future_range_1h', 'trg_future_range_3h', 'trg_future_range_6h', 'trg_future_range_24h']):
    prices[trg + '_log'] = prices[trg].apply(lambda x: np.log1p(x))

100%|██████████| 4/4 [00:04<00:00,  1.16s/it]


## Join prices to events

In [18]:
df = agg_events.merge(prices, left_on='time_to_check', right_on='time', how='left')

In [19]:
df.shape

(389482, 137)

In [20]:
def add_lagged_target_features(df, windows=[5, 20]):
    """
    Adds lagged target features for news titles and categories to capture regime shifts.
    """
    # We need to sort by date to avoid leakage
    df = df.sort_values(['instrument', 'time_to_check'])
    
    # Collect new columns in a dict to concat once per horizon
    new_cols = {}
    
    # 1. Last N events of the same dominant_event_type
    for h in tqdm([1, 3, 6, 24]):
        col = f'trg_future_range_{h}h'
        for w in windows:
            new_cols[f'mean_range_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).mean())
            ).fillna(0)
            new_cols[f'min_range_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).min())
            ).fillna(0)
            new_cols[f'max_range_{h}h_{w}'] = (
                df.groupby(['main_event', 'instrument'])[col]
                .transform(lambda x: x.shift(1).rolling(window=w, min_periods=1).max())
            ).fillna(0)
        
        new_cols[f'prev_range_{h}h'] = (
            df.groupby(['main_event', 'instrument'])[col]
            .transform(lambda x: x.shift(1))
        ).fillna(0)
    
    # Concatenate all new columns at once
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df

In [21]:
df = add_lagged_target_features(df)

100%|██████████| 4/4 [00:03<00:00,  1.19it/s]


In [22]:
df.columns[-28:]

Index(['mean_range_1h_5', 'min_range_1h_5', 'max_range_1h_5',
       'mean_range_1h_20', 'min_range_1h_20', 'max_range_1h_20',
       'prev_range_1h', 'mean_range_3h_5', 'min_range_3h_5', 'max_range_3h_5',
       'mean_range_3h_20', 'min_range_3h_20', 'max_range_3h_20',
       'prev_range_3h', 'mean_range_6h_5', 'min_range_6h_5', 'max_range_6h_5',
       'mean_range_6h_20', 'min_range_6h_20', 'max_range_6h_20',
       'prev_range_6h', 'mean_range_24h_5', 'min_range_24h_5',
       'max_range_24h_5', 'mean_range_24h_20', 'min_range_24h_20',
       'max_range_24h_20', 'prev_range_24h'],
      dtype='object')

In [23]:
df[df['main_event'] == 'No main events'].head()

,rounded_time,news_count,high_impact_count,key_event_count,main_event,prev_hour_news_count,next_hour_news_count,prev_hour_high_impact_count,next_hour_high_impact_count,prev_hour_main_event,next_hour_main_event,category_bnd,category_bsnss,category_cnsm,category_enrg,category_gdp,category_gov,category_hse,category_lbr,category_mny,category_mrkt,category_prce,category_trd,currency_AUD,currency_CAD,currency_EUR,currency_GBP,currency_JPY,currency_SEK,currency_SGD,currency_USD,country_AU,country_CA,country_DE,country_EU,country_FR,country_GB,country_JP,country_SE,country_SG,country_US,source_CENTRAL_BANK,source_ENERGY,source_GOV_MINISTRY,source_INDUSTRY,source_OFFICES,source_OFFICIAL_STATS,source_PRIVATE_SURVEY,source_RATING_AGENCY,source_RESEARCH_INSTITUTES,source_US_HOUSING,event_MONETARY_POLICY,event_CB_SPEECH,event_INFLATION,event_LABOR_MARKET,event_ECONOMIC_ACTIVITY,event_SENTIMENT,event_CONSUMER_HOUSING,event_TRADE_FINANCE,event_COMMODITIES,stage_release_Final,stage_release_Flash,stage_release_Preliminary,calc_period_MoM,calc_period_QoQ,calc_period_YoY,is_calendar,is_president,mie_Balance_of_Trade,mie_Core_Inflation_rate,mie_FOMC,mie_GDP,mie_Inflation_rate,mie_Interest_Rate_Decision,mie_NFP,mie_PMI,mie_PMI_Manufacturing,mie_PMI_Services,mie_Retail_Sales,mie_Unemployment_rate,time_passed_from_last_events,time_left_to_next_events,last_important_event_in_hours,time_to_check,time,instrument,base_currency,quote_currency,daily_atr,trange,atr,kaufman_efficiency_ratio,custom_efficiency_ratio,wick_ratio,relative_range,relative_atr,normalized_bb_width,distance_from_sma,ADX_21,ADXR_21_2,DMP_21,DMN_21,di_spread,tr_over_atr,realized_vol_ratio,parkinson_vol,parkinson_vol_over_atr,atr_short_over_long,abs_log_return_sum,vvov,week,month,quarter,dayofweek,day,hour,log_return,trg_future_range_1h,trg_future_range_3h,trg_future_range_6h,trg_future_range_24h,trg_overall_future_range_1h,trg_overall_future_range_3h,trg_overall_future_range_6h,trg_overall_future_range_24h,trg_big_doji,trg_dir_changes,trg_is_flat_flg,trg_is_trend_flg,trg_is_chaos_1h,trg_is_chaos_3h,trg_is_chaos_6h,trg_is_chaos_24h,trg_future_range_1h_log,trg_future_range_3h_log,trg_future_range_6h_log,trg_future_range_24h_log,mean_range_1h_5,min_range_1h_5,max_range_1h_5,mean_range_1h_20,min_range_1h_20,max_range_1h_20,prev_range_1h,mean_range_3h_5,min_range_3h_5,max_range_3h_5,mean_range_3h_20,min_range_3h_20,max_range_3h_20,prev_range_3h,mean_range_6h_5,min_range_6h_5,max_range_6h_5,mean_range_6h_20,min_range_6h_20,max_range_6h_20,prev_range_6h,mean_range_24h_5,min_range_24h_5,max_range_24h_5,mean_range_24h_20,min_range_24h_20,max_range_24h_20,prev_range_24h
37,2013-01-25 13:00:00+00:00,1,0,0,No main events,27.0,1.0,0.0,0.0,Unemployment_rate,No main events,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,104.0,170.0,52.0,2013-01-25 12:00:00+00:00,2013-01-25 12:00:00+00:00,AUDCAD,AUD,CAD,0.005500,0.00124,0.001311,0.007636,6.470588,0.330645,-0.038678,0.481734,4.552950,5.117688,13.861417,14.241713,0.004829,0.004113,0.000715,0.945962,0.360951,0.000800,0.609956,0.944024,0.001917,0.047268,4,1.0,1.0,4.0,25.0,12.0,-0.000382,3.234582,3.776222,4.447550,5.477429,1.255022,2.092396,3.341764,9.051664,1.0,1.0,1.0,0.0,False,False,False,True,1.443285,1.563650,1.695166,1.868324,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
49,2013-01-29 02:00:00+00:00,1,0,0,No main events,1.0,2.0,0.0,1.0,No main events,NFP,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,170.0,166.0,85.0,2013-01-29 01:00:00+00:00,2013-01-29 01:00:00+00:00,AUDCAD,AUD,CAD,0.005548,0.00165,0.001399,0.043420,5.836935,0.618182,0.373413,0.552003,5.108678,3.897428,12

In [24]:
df.isna().sum().sort_values(ascending=False)[:30]

wick_ratio                  1210
trg_is_chaos_1h             1201
trg_is_chaos_3h             1201
trg_is_chaos_6h             1201
trg_is_trend_flg            1201
trg_future_range_1h_log     1201
trg_future_range_3h_log     1201
trg_future_range_6h_log     1201
trg_future_range_24h_log    1201
parkinson_vol               1201
parkinson_vol_over_atr      1201
atr_short_over_long         1201
abs_log_return_sum          1201
vvov                        1201
week                        1201
month                       1201
quarter                     1201
dayofweek                   1201
day                         1201
hour                        1201
DMP_21                      1201
daily_atr                   1201
time                        1201
ADX_21                      1201
ADXR_21_2                   1201
normalized_bb_width         1201
relative_atr                1201
relative_range              1201
distance_from_sma           1201
base_currency               1201
dtype: int

In [25]:
df.dropna(subset=[col for col in df.columns if col.startswith('trg_')] + ['wick_ratio'], inplace=True)

In [26]:
df.shape

(388272, 165)

In [27]:
df.isna().sum().sort_values(ascending=False)

next_hour_main_event             12
prev_hour_main_event             12
last_important_event_in_hours    12
news_count                        0
high_impact_count                 0
                                 ..
max_range_24h_5                   0
mean_range_24h_20                 0
min_range_24h_20                  0
max_range_24h_20                  0
prev_range_24h                    0
Length: 165, dtype: int64

In [28]:
for col in ['prev_hour_main_event', 'next_hour_main_event', 'last_important_event_in_hours']:
    df[col] = df[col].where(pd.notnull(df[col]), None)
#df.dropna(subset=['prev_hour_main_event', 'next_hour_main_event', 'last_important_event_in_hours'], inplace=True)

# Train models

In [29]:
SUFFX = 'Fixed ranges for No main events'

In [30]:
df_full = df.copy()

In [31]:
df = df_full.copy()

In [32]:
df.drop('daily_atr', axis=1, inplace=True)

In [33]:
FEATURES = [i for i in df.columns if not i.startswith('trg') and i not in ['time', 'time_to_check', 'rounded_time']]
CATEGORICAL_FEATURES = ['main_event', 'instrument', 'base_currency', 'quote_currency', 'prev_hour_main_event', 'next_hour_main_event']

In [34]:
# df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].fillna(None)

# for col in CATEGORICAL_FEATURES:
#     df[col] = df[col].where(pd.notnull(df[col]), None)

df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].fillna("missing").astype(str)

for cat_feat in CATEGORICAL_FEATURES:
    df[cat_feat] = df[cat_feat].astype('category')

In [35]:
print(f'TARGETS: {[i for i in df.columns if i.startswith('trg')]}')

TARGETS: ['trg_future_range_1h', 'trg_future_range_3h', 'trg_future_range_6h', 'trg_future_range_24h', 'trg_overall_future_range_1h', 'trg_overall_future_range_3h', 'trg_overall_future_range_6h', 'trg_overall_future_range_24h', 'trg_big_doji', 'trg_dir_changes', 'trg_is_flat_flg', 'trg_is_trend_flg', 'trg_is_chaos_1h', 'trg_is_chaos_3h', 'trg_is_chaos_6h', 'trg_is_chaos_24h', 'trg_future_range_1h_log', 'trg_future_range_3h_log', 'trg_future_range_6h_log', 'trg_future_range_24h_log']


## Quantile ranges
- predict ranges for the next 1, 3, 6, 24 hours

In [36]:
from training_functions import (
    run_time_series_cv_catboost_quantile_gpu, 
    run_time_series_cv_catboost_quantile, 
    run_time_series_cv_xgboost_quantile, 
    run_time_series_cv_xgboost_multi_quantile,
    run_time_series_cv_lightgbm_multi_quantile
)

In [37]:
DESC = f"""
{SUFFX}
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


Fixed ranges for No main events
- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [42]:
EXPERIMENT_NAME = 'Target: Total Range Prediction || Model: XGBoost'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_xgboost_multi_quantile(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour


2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/05/10 21:56:49 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/05/10 21:56:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/10 21:56:49 INFO mlflow.store.db.utils: Updating database tables
2026/05/10 21:56:49 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/05/10 21:56:49 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/05/10 21:56:49 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/05/10 21:56:49 INFO alembic.runtime

[INFO] Train model for 3 hour


[INFO] Train model for 6 hour


[INFO] Train model for 24 hour


In [43]:
EXPERIMENT_NAME = 'Target: Total Range Prediction || Model: LightGBM'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_lightgbm_multi_quantile(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour


2026/05/10 22:31:14 INFO mlflow.tracking.fluent: Experiment with name 'Target: Total Range Prediction || Model: LightGBM' does not exist. Creating a new experiment.
lgb mq CV:   0%|          | 0/5 [00:00<?, ?it/s]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.9: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
lgb mq CV:  20%|██        | 1/5 [00:43<02:55, 43.87s/it]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake

[INFO] Train model for 3 hour


lgb mq CV:   0%|          | 0/5 [00:00<?, ?it/s]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.9: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
lgb mq CV:  20%|██        | 1/5 [00:35<02:23, 35.89s/it]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.


[INFO] Train model for 6 hour


lgb mq CV:   0%|          | 0/5 [00:00<?, ?it/s]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.9: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
lgb mq CV:  20%|██        | 1/5 [00:36<02:27, 36.84s/it]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.


[INFO] Train model for 24 hour


lgb mq CV:   0%|          | 0/5 [00:00<?, ?it/s]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.9: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
lgb mq CV:  20%|██        | 1/5 [00:36<02:25, 36.47s/it]<string>:33: UserWarning: LightGBM quantile α=0.1: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_CUDA=1); retrying CPU.
<string>:33: UserWarning: LightGBM quantile α=0.5: CUDA failed (LightGBMError: CUDA Tree Learner was not enabled in this build.


In [39]:
EXPERIMENT_NAME = 'Target: Total Range Prediction || Model: CatBoost'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour


2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/05/24 12:12:30 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/05/24 12:12:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 12:12:30 INFO mlflow.store.db.utils: Updating database tables
2026/05/24 12:12:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/05/24 12:12:30 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/05/24 12:12:30 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/05/24 12:12:30 INFO alembic.runtime


Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


In [38]:
# EXPERIMENT_NAME = 'Target: Total Range Prediction || TF: 1h || Price source: MT5 + TW || [Time cropping: 1st method]'

# hours = [1, 3, 6, 24]
# for hour in hours:
#     print(f'[INFO] Train model for {hour} hour')
#     TARGET = f'trg_future_range_{hour}h'
#     RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
#     metrics = run_time_series_cv_catboost_quantile_gpu(
#         df=df,
#         features=FEATURES,
#         cat_features=CATEGORICAL_FEATURES,
#         target=TARGET,
#         time_col='time_to_check',
#         quantiles=[0.1, 0.5, 0.9],
#         experiment_name=EXPERIMENT_NAME,
#         run_name=RUN_NAME,
#         description=DESC,
#         model_params={
#             'iterations': 5000,
#             'learning_rate': 0.05
#         },
#         verbose=False,
#         save_model=True,
#         model_name=f'range_prediction_model_{hour}h'
#     )

In [40]:
EXPERIMENT_NAME = 'Target: Log Total Range Prediction || Model: CatBoost'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_future_range_{hour}h_log'
    RUN_NAME = f'total_range_prediction_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'log_range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour



Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


In [41]:
EXPERIMENT_NAME = 'Target: Sum of Ranges Prediction || Model: CatBoost'

hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_overall_future_range_{hour}h'
    RUN_NAME = f'trg_overall_future_range_{hour}h_{SUFFX}'
    metrics = run_time_series_cv_catboost_quantile_gpu(
        df=df,
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        quantiles=[0.1, 0.5, 0.9],
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        verbose=False,
        save_model=True,
        model_name=f'sum_range_prediction_model_{hour}h'
    )

[INFO] Train model for 1 hour



Training final GPU quantile models on all data...
[INFO] Train model for 3 hour



Training final GPU quantile models on all data...
[INFO] Train model for 6 hour



Training final GPU quantile models on all data...
[INFO] Train model for 24 hour



Training final GPU quantile models on all data...


## Direction changes count

In [42]:
from training_functions import run_time_series_cv_catboost

In [43]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [44]:
EXPERIMENT_NAME = 'Target: Direction Count || Model: CatBoost'

print(f'[INFO] Train model for Direction Count')
TARGET = f'trg_dir_changes'
RUN_NAME = f'trg_direction_count_{SUFFX}'
metrics = run_time_series_cv_catboost(
    df=df,
    task_type='ordinal',
    features=FEATURES,
    cat_features=CATEGORICAL_FEATURES,
    target=TARGET,
    time_col='time_to_check',
    experiment_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    description=DESC,
    model_params={
        'iterations': 5000,
        'learning_rate': 0.05
    },
    save_model=True,
    model_name=f'dir_changes_model_{hour}h'
)

[INFO] Train model for Direction Count



Training final model on all data...

Final model metrics (full data, in-sample):
  MAE: 0.7440
  RMSE: 0.9256
  MAE_rounded: 0.7074
  Within1: 0.9011
  Spearman: 0.5650
  QuadraticKappa: 0.4374


## Chaos prediction

In [45]:
from training_functions import run_time_series_cv_catboost

In [46]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [47]:
EXPERIMENT_NAME = 'Target: Chaos prediction || Model: CatBoost'

print(f'[INFO] Train model for Chaos Prediction')
hours = [1, 3, 6, 24]
for hour in hours:
    print(f'[INFO] Train model for {hour} hour')
    TARGET = f'trg_is_chaos_{hour}h'
    RUN_NAME = f'trg_is_chaos_{hour}h_{SUFFX}'
    df[TARGET] = df[TARGET].astype(int)
    metrics = run_time_series_cv_catboost(
        df=df,
        task_type='classification',
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        },
        save_model=True,
        model_name=f'chaos_prediction_model_{hour}h'
    )

[INFO] Train model for Chaos Prediction
[INFO] Train model for 1 hour



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.7760
  AUC:       0.8233

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.773436  0.994018  0.869962  292696.000000
1             1   0.855289  0.108280  0.192225   95576.000000
2      accuracy   0.775987  0.775987  0.775987       0.775987
3     macro avg   0.814363  0.551149  0.531094  388272.000000
4  weighted avg   0.793585  0.775987  0.703132  388272.000000
[INFO] Train model for 3 hour



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.8222
  AUC:       0.8656

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.822677  0.976770  0.893126  295346.000000
1             1   0.817556  0.330855  0.471072   92926.000000
2      accuracy   0.822181  0.822181  0.822181       0.822181
3     macro avg   0.820117  0.653812  0.682099  388272.000000
4  weighted avg   0.821452  0.822181  0.792115  388272.000000
[INFO] Train model for 6 hour



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.8415
  AUC:       0.8918

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.843966  0.970557  0.902845  294602.000000
1             1   0.824700  0.435646  0.570125   93670.000000
2      accuracy   0.841511  0.841511  0.841511       0.841511
3     macro avg   0.834333  0.703102  0.736485  388272.000000
4  weighted avg   0.839318  0.841511  0.822577  388272.000000
[INFO] Train model for 24 hour



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.8870
  AUC:       0.9454

Classification Report:
          index  precision    recall  f1-score       support
0             0   0.891802  0.965536  0.927205  289523.00000
1             1   0.866624  0.656543  0.747096   98749.00000
2      accuracy   0.886950  0.886950  0.886950       0.88695
3     macro avg   0.879213  0.811040  0.837151  388272.00000
4  weighted avg   0.885398  0.886950  0.881398  388272.00000


## Trend / Flat prediction

In [48]:
from training_functions import run_time_series_cv_catboost

In [49]:
DESC = f"""
- One hot classes: {[i for i in df.columns if i.startswith('category')]}
- One hot events: {[i for i in df.columns if i.startswith('event')]}
- One hot sources: {[i for i in df.columns if i.startswith('source')]}
- Main aggregations: {['news_count', 'high_impact_count', 'key_event_count', 'main_event']}
- Currency: {[i for i in df.columns if i.startswith('currency')]}
"""
print(DESC)


- One hot classes: ['category_bnd', 'category_bsnss', 'category_cnsm', 'category_enrg', 'category_gdp', 'category_gov', 'category_hse', 'category_lbr', 'category_mny', 'category_mrkt', 'category_prce', 'category_trd']
- One hot events: ['event_MONETARY_POLICY', 'event_CB_SPEECH', 'event_INFLATION', 'event_LABOR_MARKET', 'event_ECONOMIC_ACTIVITY', 'event_SENTIMENT', 'event_CONSUMER_HOUSING', 'event_TRADE_FINANCE', 'event_COMMODITIES']
- One hot sources: ['source_CENTRAL_BANK', 'source_ENERGY', 'source_GOV_MINISTRY', 'source_INDUSTRY', 'source_OFFICES', 'source_OFFICIAL_STATS', 'source_PRIVATE_SURVEY', 'source_RATING_AGENCY', 'source_RESEARCH_INSTITUTES', 'source_US_HOUSING']
- Main aggregations: ['news_count', 'high_impact_count', 'key_event_count', 'main_event']
- Currency: ['currency_AUD', 'currency_CAD', 'currency_EUR', 'currency_GBP', 'currency_JPY', 'currency_SEK', 'currency_SGD', 'currency_USD']



In [50]:
EXPERIMENT_NAME = 'Target: Flat / Trend prediction || Model: CatBoost'

print(f'[INFO] Train model for Flat / Trend prediction')
regime = ['trg_is_flat_flg', 'trg_is_trend_flg']
for r in regime:
    print(f'[INFO] Train model for {r}')
    TARGET = r
    RUN_NAME = f'trg_{r}_prediction_{SUFFX}'
    df[TARGET] = df[TARGET].astype(int)
    metrics = run_time_series_cv_catboost(
        df=df,
        task_type='classification',
        features=FEATURES,
        cat_features=CATEGORICAL_FEATURES,
        target=TARGET,
        time_col='time_to_check',
        experiment_name=EXPERIMENT_NAME,
        run_name=RUN_NAME,
        description=DESC,
        model_params={
            'iterations': 5000,
            'learning_rate': 0.05
        }
    )

[INFO] Train model for Flat / Trend prediction
[INFO] Train model for trg_is_flat_flg



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.8689
  AUC:       0.9454

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.865870  0.901152  0.883159  213439.000000
1             1   0.873008  0.829580  0.850740  174833.000000
2      accuracy   0.868924  0.868924  0.868924       0.868924
3     macro avg   0.869439  0.865366  0.866950  388272.000000
4  weighted avg   0.869084  0.868924  0.868561  388272.000000
[INFO] Train model for trg_is_trend_flg



Training final model on all data...

Final Model Metrics (on full dataset):
  Accuracy:  0.8947
  AUC:       0.9618

Classification Report:
          index  precision    recall  f1-score        support
0             0   0.883056  0.942518  0.911819  224245.000000
1             1   0.913448  0.829357  0.869374  164027.000000
2      accuracy   0.894713  0.894713  0.894713       0.894713
3     macro avg   0.898252  0.885938  0.890596  388272.000000
4  weighted avg   0.895895  0.894713  0.893888  388272.000000
